# 06 - Second model: LLaVA-1.5-7B

Tests whether the directional bias generalizes beyond Qwen2.5-VL.

**It does not - but for an instructive reason.** LLaVA reproduces the behavioral
gap (0.844 -> 0.608, a 24-point drop that looks like a replication) while failing
by an entirely different mechanism: it cannot resolve the rendered text at all.
Transcription median is 0.039 at 5pt and never exceeds 0.23 at any font size,
because LLaVA-1.5's fixed 336x336 CLIP encoder downsamples the canvas past
legibility regardless of type size.

So the two models produce indistinguishable accuracy drops from *perception loss*
and *readout failure* respectively. That contrast is the argument for opening the
model, and the transcription check is what separates them.

Does not need Qwen loaded. Produces `results/replication/`.

> Reconstructed from session transcripts; outputs not embedded.

In [ ]:
from kaggle_secrets import UserSecretsClient
token = UserSecretsClient().get_secret("GH_TOKEN")
import os, sys
if not os.path.isdir("/kaggle/working/algoverse"):
    !git clone https://{token}@github.com/bryantran21/algoverse.git /kaggle/working/algoverse
sys.path.insert(0, "/kaggle/working/algoverse"); os.chdir("/kaggle/working/algoverse")
!git config user.email "bryantran21@gmail.com"
!git config user.name  "bryantran21"

import subprocess
subprocess.run([sys.executable,'-m','pip','install','-q','transformers>=4.49.0',
    'accelerate>=0.34.0','datasets','qwen-vl-utils','typst','Pillow',
    'scikit-learn','matplotlib','tqdm'], check=True)

import torch, numpy as np, pickle, os, config
assert torch.cuda.is_available(), "NO GPU - set Accelerator to T4 x2"
os.makedirs("results/replication", exist_ok=True)

from transformers import LlavaForConditionalGeneration, AutoProcessor
MID = "llava-hf/llava-1.5-7b-hf"
model2 = LlavaForConditionalGeneration.from_pretrained(
    MID, torch_dtype=torch.bfloat16, device_map="auto")
proc2 = AutoProcessor.from_pretrained(MID)
print("LLaVA loaded", flush=True)

# answer token ids are model-specific - never reuse Qwen's
for w in ["Yes", "No", " Yes", " No"]:
    print(repr(w), proc2.tokenizer.encode(w, add_special_tokens=False), flush=True)
YES2, NO2 = 3869, 1939

## Per-class accuracy at 5pt

LLaVA needs its own prompt format (`USER: <image>\n... ASSISTANT:`), so this does
not reuse `_messages_for` for prompt construction - only for the render.

In [ ]:
from src.inference import load_items, _messages_for, _flatten_images
config.RENDER["font_size_pt"] = 5.0
items2 = load_items(n=500)

@torch.no_grad()
def yn2(item, mode):
    if mode == "image":
        _, images = _messages_for(item, "image")
        img = _flatten_images([images])[0]
        prompt = f"USER: <image>\nQuestion: {item['question']}\nAnswer yes or no.\nASSISTANT:"
        inp = proc2(text=prompt, images=img, return_tensors="pt").to(model2.device)
    else:
        q = f"{item['passage']}\n\nQuestion: {item['question']}\nAnswer yes or no."
        inp = proc2(text=f"USER: {q}\nASSISTANT:", return_tensors="pt").to(model2.device)
    lg = model2(**inp, use_cache=False).logits[0, -1].float()
    return lg[YES2].item(), lg[NO2].item()

rows2 = []
for k, it in enumerate(items2):
    ty, tn = yn2(it, "text"); iy, inn = yn2(it, "image")
    rows2.append(dict(item_id=it["item_id"], gold=it["gold"],
                      d_txt=ty-tn, d_img=iy-inn))
    if (k+1) % 100 == 0:
        pickle.dump(rows2, open("results/replication/llava_logits.pkl","wb"))
        print(f"{k+1}/{len(items2)}", flush=True)
pickle.dump(rows2, open("results/replication/llava_logits.pkl","wb"))

g  = np.array([1 if r["gold"]=="yes" else 0 for r in rows2])
at = np.array([(r["d_txt"]>0)==bool(b) for r,b in zip(rows2, g)])
ai = np.array([(r["d_img"]>0)==bool(b) for r,b in zip(rows2, g)])
s  = np.array([r["d_img"]-r["d_txt"] for r in rows2])
err = ~ai
print(f"\nLLaVA text {at.mean():.3f}  image {ai.mean():.3f}")
print(f"gold-yes: text {at[g==1].mean():.3f} image {ai[g==1].mean():.3f}")
print(f"gold-no : text {at[g==0].mean():.3f} image {ai[g==0].mean():.3f}")
print(f"false-no rate {(g[err]==1).mean():.3f}   shift {s.mean():+.3f}")

## Legibility check - the essential control

Symmetric degradation is ambiguous: it is what you would see both from a model
that reads the text and weighs it poorly, *and* from a model that cannot read the
text and is guessing. Transcription distinguishes them.

Run this before interpreting any accuracy number from a new model.

In [ ]:
import difflib, re
norm = lambda s: re.sub(r"\s+"," ",s.lower().strip())

@torch.no_grad()
def transcribe2(item):
    _, images = _messages_for(item, "image")
    img = _flatten_images([images])[0]
    prompt = "USER: <image>\nTranscribe all text in this image exactly. Output only the text.\nASSISTANT:"
    inp = proc2(text=prompt, images=img, return_tensors="pt").to(model2.device)
    out = model2.generate(**inp, max_new_tokens=512, do_sample=False)
    return proc2.decode(out[0][inp["input_ids"].shape[1]:], skip_special_tokens=True).strip()

sc = np.array([difflib.SequenceMatcher(
        None, norm(transcribe2(it)), norm(f"{it['passage']} {it['question']}")
      ).ratio() for it in items2[:40]])
print(f"LLaVA transcription: mean {sc.mean():.3f} median {np.median(sc):.3f} (Qwen was 0.96 median)")

## Font sweep - is legibility the limiting factor?

If LLaVA simply needs bigger text, the bias hypothesis could still be tested at a
larger size. It does not: transcription peaks at 0.233 (8pt) and *declines* by
14pt, because the bottleneck is the fixed 336x336 encoder rather than type size -
and larger text fits less of the passage on the page.

Accuracy is flat across all three sizes, gold-yes tracks the text-mode value, and
gold-no collapses: the signature of a model answering "yes" while ignoring the
image.

In [ ]:
FONTS = [8.0, 11.0, 14.0]
llava_sweep = {}

d_txt = {}
for it in items2:
    ty, tn = yn2(it, "text")
    d_txt[it["item_id"]] = ty - tn
print("text baseline done", flush=True)

for fs in FONTS:
    config.RENDER["font_size_pt"] = fs
    sc = np.array([difflib.SequenceMatcher(
            None, norm(transcribe2(it)), norm(f"{it['passage']} {it['question']}")
          ).ratio() for it in items2[:30]])

    rows = []
    for it in items2:
        iy, inn = yn2(it, "image")
        rows.append(dict(item_id=it["item_id"], gold=it["gold"],
                         d_txt=d_txt[it["item_id"]], d_img=iy-inn))

    g   = np.array([1 if r["gold"]=="yes" else 0 for r in rows])
    ai  = np.array([(r["d_img"]>0)==bool(b) for r,b in zip(rows, g)])
    at  = np.array([(r["d_txt"]>0)==bool(b) for r,b in zip(rows, g)])
    sh  = np.array([r["d_img"]-r["d_txt"] for r in rows])
    err = ~ai

    llava_sweep[fs] = dict(
        transcription=float(np.median(sc)),
        acc_img=float(ai.mean()), acc_txt=float(at.mean()),
        yes_img=float(ai[g==1].mean()), no_img=float(ai[g==0].mean()),
        yes_txt=float(at[g==1].mean()), no_txt=float(at[g==0].mean()),
        false_no=float((g[err]==1).mean()), shift=float(sh.mean()),
        shift_sd=float(sh.std()), n=len(rows))

    print(f"{fs:5.1f}pt  transcr {np.median(sc):.3f} | img acc {ai.mean():.3f}  "
          f"yes {ai[g==1].mean():.3f} (txt {at[g==1].mean():.3f})  "
          f"no {ai[g==0].mean():.3f} (txt {at[g==0].mean():.3f})  "
          f"false-no {(g[err]==1).mean():.3f}  shift {sh.mean():+.3f}", flush=True)
    pickle.dump(llava_sweep, open("results/replication/llava_font_sweep.pkl","wb"))

config.RENDER["font_size_pt"] = 5.0

In [ ]:
!git add -A && git commit -m "06: LLaVA-1.5-7B replication and legibility-conditioned font sweep" && git push origin master